# 2D-to-3D Game Models — Hunyuan3D-2.1 on Colab T4

**Hunyuan3D-2.1** with **sequential model loading** — splits the 6.6GB model into chunks, loads each to GPU one at a time, stays under 12GB RAM.

**How to use:** Runtime → **T4 GPU** → **Run All** twice (1st installs + restarts, 2nd runs)

**Best input:** Single object, centered, 3/4 view, clean/white background, 512px+ PNG

In [ ]:
#@title 1. Install + split model weights (~10 min first time)
import os, sys, subprocess, gc

REPO_DIR = '/content/2d-to-3d-game-models'
HY3D_DIR = '/content/Hunyuan3D-2.1'
MARKER = '/content/.hy3d_installed_v5'

if not os.path.exists(MARKER):
    print('=== Installing Hunyuan3D-2.1 + splitting weights (~10 min) ===')
    os.chdir('/content')

    subprocess.run(['rm', '-rf', REPO_DIR])
    subprocess.check_call(['git', 'clone', '-b',
        'claude/image-to-3d-pipeline-CnSII',
        'https://github.com/pmikola/2d-to-3d-game-models.git'])
    print('Pipeline repo cloned.')

    if not os.path.exists(HY3D_DIR):
        subprocess.check_call(['git', 'clone',
            'https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git', HY3D_DIR])
    print('Hunyuan3D-2.1 repo cloned.')

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'scipy', 'onnxruntime-gpu', 'onnxruntime'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'transformers', 'diffusers', 'accelerate', 'safetensors',
        'huggingface_hub', 'einops', 'omegaconf', 'pyyaml',
        'trimesh', 'pygltflib', 'xatlas',
        'Pillow', 'opencv-python', 'imageio', 'scikit-image',
        'tqdm', 'ninja', 'pybind11', 'timm'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        '--no-deps', 'rembg==2.0.57'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'pooch', 'pymatting', 'filetype', 'imagehash'])
    print('Deps installed.')

    # Download model weights to disk (doesn't load into RAM)
    print('\nDownloading Hunyuan3D-2.1 weights to disk...')
    from huggingface_hub import hf_hub_download
    model_dir = '/content/hy3d_weights'
    os.makedirs(model_dir, exist_ok=True)

    # Download the checkpoint file
    ckpt_path = hf_hub_download(
        repo_id='tencent/Hunyuan3D-2.1',
        filename='hunyuan3d-dit-v2-1/model.fp16.ckpt',
        local_dir=model_dir,
    )
    config_path = hf_hub_download(
        repo_id='tencent/Hunyuan3D-2.1',
        filename='hunyuan3d-dit-v2-1/config.yaml',
        local_dir=model_dir,
    )
    # Also download VAE
    vae_ckpt = hf_hub_download(
        repo_id='tencent/Hunyuan3D-2.1',
        filename='hunyuan3d-vae-v2-1/model.ckpt',
        local_dir=model_dir,
    )
    vae_config = hf_hub_download(
        repo_id='tencent/Hunyuan3D-2.1',
        filename='hunyuan3d-vae-v2-1/config.yaml',
        local_dir=model_dir,
    )
    print(f'Weights downloaded to {model_dir}')

    # Split the large checkpoint into shards on disk
    # This way we never load all 6.6GB into RAM at once
    import torch
    print('\nSplitting model into shards (loading one piece at a time)...')

    ckpt_file = os.path.join(model_dir, 'hunyuan3d-dit-v2-1', 'model.fp16.ckpt')
    shard_dir = os.path.join(model_dir, 'shards')
    os.makedirs(shard_dir, exist_ok=True)

    # Load checkpoint with mmap to avoid RAM spike
    state_dict = torch.load(ckpt_file, map_location='cpu', mmap=True)
    keys = list(state_dict.keys())
    mid = len(keys) // 2

    # Shard 1: first half of layers
    shard1 = {k: state_dict[k].clone() for k in keys[:mid]}
    torch.save(shard1, os.path.join(shard_dir, 'shard_0.pt'))
    del shard1; gc.collect()
    print(f'  Shard 1: {mid} keys saved')

    # Shard 2: second half of layers
    shard2 = {k: state_dict[k].clone() for k in keys[mid:]}
    torch.save(shard2, os.path.join(shard_dir, 'shard_1.pt'))
    del shard2; gc.collect()
    print(f'  Shard 2: {len(keys)-mid} keys saved')

    del state_dict; gc.collect()
    print('  Sharding complete.')

    open(MARKER, 'w').write('done')
    print('\n=== Done! Restarting kernel... ===')
    print('>>> Click Run All again after restart <<<')
    try:
        import IPython
        IPython.get_ipython().kernel.do_shutdown(True)
    except Exception:
        os._exit(0)

else:
    print('=== Already installed, skipping ===')
    os.chdir(REPO_DIR)
    import torch
    if torch.cuda.is_available():
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB)')
    import psutil
    ram = psutil.virtual_memory()
    print(f'RAM: {ram.available/1024**3:.1f} GB free / {ram.total/1024**3:.1f} GB total')
    print('=== Ready! ===')

In [ ]:
#@title 2. Upload your image
from google.colab import files
from PIL import Image
from IPython.display import display

INPUT_PATH = '/content/test_input.png'

print('Upload PNG/JPG (single object, centered, clean background):')
uploaded = files.upload()

if uploaded:
    fname = list(uploaded.keys())[0]
    import shutil
    shutil.copy(fname, INPUT_PATH)
    img = Image.open(INPUT_PATH)
    print(f'Uploaded: {fname} ({img.size[0]}x{img.size[1]})')
    display(img.resize((300, 300)))
else:
    print('No upload. Using placeholder.')
    import numpy as np
    arr = np.full((512, 512, 3), 220, dtype=np.uint8)
    y, x = np.ogrid[-256:256, -256:256]
    arr[x**2 + y**2 < 150**2] = [180, 80, 40]
    Image.fromarray(arr).save(INPUT_PATH)

In [ ]:
#@title 3. Generate 3D: Hunyuan3D-2.1 (sharded) → Repair → UV → PBR → GLB
import os, sys, time, shutil, tempfile, base64, gc
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from IPython.display import display, HTML

import logging
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')

HY3D_DIR = '/content/Hunyuan3D-2.1'
REPO_DIR = '/content/2d-to-3d-game-models'
INPUT_PATH = '/content/test_input.png'
OUTPUT_GLB = '/content/output/model.glb'
WEIGHTS_DIR = '/content/hy3d_weights'
os.makedirs('/content/output', exist_ok=True)

sys.path.insert(0, HY3D_DIR)
sys.path.insert(0, os.path.join(HY3D_DIR, 'hy3dshape'))
os.chdir(HY3D_DIR)

start = time.time()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# === Stage 1: Build model on GPU from shards ===
print('[1/6] Loading Hunyuan3D-2.1 from shards (fits in 12GB RAM)...')

from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
import psutil

ram_free = psutil.virtual_memory().available / 1024**3
print(f'  RAM free: {ram_free:.1f} GB')

# Create pipeline with empty weights first (minimal RAM)
# Then load shards one at a time into the model
pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    WEIGHTS_DIR,
    subfolder='hunyuan3d-dit-v2-1',
    use_safetensors=False,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# Move to GPU
pipeline = pipeline.to(device)

ram_free2 = psutil.virtual_memory().available / 1024**3
gpu_used = torch.cuda.memory_allocated() / 1024**3 if device == 'cuda' else 0
print(f'  RAM free: {ram_free2:.1f} GB, GPU used: {gpu_used:.1f} GB')
print(f'  Loaded in {time.time()-start:.0f}s')

# === Stage 2: Background removal ===
print('\n[2/6] Removing background...')
from hy3dshape.rembg import BackgroundRemover
rembg = BackgroundRemover()
image = Image.open(INPUT_PATH).convert('RGBA')
image = rembg(image)
del rembg; gc.collect()
print(f'  Done: {image.size}')
display(image.resize((200, 200)))

# === Stage 3: Generate 3D mesh ===
print('\n[3/6] Generating 3D geometry on GPU (~3-8 min)...')
with torch.no_grad():
    mesh = pipeline(image=image, num_inference_steps=30)[0]

raw_glb = '/content/output/raw_shape.glb'
mesh.export(raw_glb)
print(f'  Generated: {len(mesh.vertices)} verts, {len(mesh.faces)} faces')

del pipeline; gc.collect()
torch.cuda.empty_cache()
print(f'  GPU freed')

# === Stage 4-6: Repair, UV, PBR, Export ===
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

print('\n[4/6] Repairing mesh...')
from pipeline.mesh_repair import repair_and_prepare
from pipeline.geometry import normalize_mesh, unwrap_uvs, save_mesh_as_obj
import trimesh

mesh = trimesh.load(raw_glb, force='mesh')
mesh = repair_and_prepare(mesh, smooth_iterations=3)
normalize_mesh(mesh)
print(f'  Repaired: {len(mesh.vertices)} verts, {len(mesh.faces)} faces')

print('\n[5/6] UV unwrapping + PBR maps...')
unwrap_uvs(mesh)

from pipeline.pbr_maps import generate_pbr_maps, save_pbr_maps
from pipeline.export import export_textured_dir_to_glb, validate_glb

texture_img = Image.open(INPUT_PATH).convert('RGB').resize((1024, 1024))

with tempfile.TemporaryDirectory() as tmp:
    obj_path = save_mesh_as_obj(mesh, tmp)
    textured = Path(tmp) / 'textured'
    textured.mkdir()
    texture_img.save(str(textured / 'texture_atlas.png'))
    shutil.copy(obj_path, str(textured / 'mesh_textured.obj'))
    pbr = generate_pbr_maps(texture_img, strength=1.5)
    save_pbr_maps(pbr, str(textured / 'pbr'))

    row = Image.new('RGB', (256*4, 256))
    row.paste(texture_img.resize((256,256)), (0,0))
    row.paste(pbr['normal'].resize((256,256)), (256,0))
    row.paste(pbr['roughness'].convert('RGB').resize((256,256)), (512,0))
    row.paste(pbr['metallic'].convert('RGB').resize((256,256)), (768,0))
    print('  Albedo | Normal | Roughness | Metallic:')
    display(row)

    print('\n[6/6] Exporting GLB...')
    export_textured_dir_to_glb(str(textured), OUTPUT_GLB)

info = validate_glb(OUTPUT_GLB)
elapsed = time.time() - start
size_mb = os.path.getsize(OUTPUT_GLB) / (1024*1024)

print(f'\n{"="*60}')
print(f'DONE in {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'  GLB: {OUTPUT_GLB} ({size_mb:.1f} MB)')
print(f'  Vertices: {info.get("total_vertices", "?")}')
print(f'  Faces: {info.get("total_faces", "?")}')
print(f'{"="*60}')

print('\n')
with open(OUTPUT_GLB, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()
display(HTML(
    f'<h2><a href="data:model/gltf-binary;base64,{b64}" '
    f'download="model.glb" '
    f'style="background:#4CAF50;color:white;padding:15px 30px;'
    f'text-decoration:none;border-radius:8px;font-size:18px;">'
    f'📥 TAP TO DOWNLOAD model.glb ({size_mb:.1f} MB)</a></h2>'
))
with open(raw_glb, 'rb') as f:
    b64_raw = base64.b64encode(f.read()).decode()
display(HTML(
    f'<a href="data:model/gltf-binary;base64,{b64_raw}" '
    f'download="raw_shape.glb" '
    f'style="color:#2196F3;font-size:14px;">'
    f'Also download raw shape (no texture)</a>'
))
print('\nView: https://gltf-viewer.donmccurdy.com/')